# 05 — RAG clásico (baseline denso, lector vía Ollama)

Retrieval denso top-5 con all-MiniLM-L6-v2 + lector vía **ChatOllama** (`READER_OLLAMA_MODEL`: `qwen2.5:1.5b` para paridad con el baseline en CPU, `qwen2.5:7b` para el run con modelo más grande) + mismo scorer. El retrieval corre igual que en local (`rag_baseline/retrieve.py`); el lector es el mismo prompt y el mismo `condense()` de `answer.py`, solo cambia el modelo que contesta (Ollama en vez de transformers en CPU).

**Prerrequisito:** notebook 01 (chunks). En GPU el embedder usa CUDA solo.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
READER_OLLAMA_MODEL = "qwen2.5:1.5b"   # lector: 1.5b (paridad CPU) o qwen2.5:7b (mas grande)
K = 5
RETRIEVAL_FILE = REPO + "/rag_baseline/retrieval.json"
TRACE_OUT = REPO + "/rag_baseline/run_trace_colab.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=READER_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", READER_OLLAMA_MODEL)

## 1. Retrieval denso top-5

Corre `rag_baseline/retrieve.py` tal cual: embebe las 151 páginas con all-MiniLM-L6-v2 y recupera top-5 por pregunta, con el diagnóstico de cobertura de cifras esperadas (separa fallos de retrieval de fallos de lector).

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "retrieve.py", "--k", str(K),
                    "--out", RETRIEVAL_FILE], cwd=REPO + "/rag_baseline")
print("exit:", r.returncode)

## 2. Lector vía Ollama

Mismo sistema y mismo `condense()` que `rag_baseline/answer.py`: el contexto son los 5 chunks recuperados (ventanas de ~1,800 caracteres por chunk, 2 ventanas) y el modelo responde solo desde ellos. La traza tiene el mismo formato que la de CPU, así que el scorer no cambia.

In [ ]:
import json, sys
sys.path.insert(0, REPO + "/rag_baseline")
from answer import SYSTEM, condense
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

reader = ChatOllama(model=READER_OLLAMA_MODEL, base_url="http://localhost:11434",
                    temperature=0, num_predict=100)
print("lector RAG (Ollama):", READER_OLLAMA_MODEL)

data = json.load(open(RETRIEVAL_FILE))
with open(TRACE_OUT, "w") as f:
    for item in data["questions"]:
        condensed = [condense(r["text"], item["question"]) for r in item["retrieved"]]
        ctx = "\n\n".join(f"[Página {r['page']}]\n{c}"
                          for r, c in zip(item["retrieved"], condensed))
        user = (f"Fragmentos del informe anual de FEMSA 2024:\n\n{ctx}\n\n"
                f"Pregunta: {item['question']}")
        ans = reader.invoke([SystemMessage(content=SYSTEM),
                             HumanMessage(content=user)]).content.strip()
        rec = {"id": item["id"], "level": item["level"],
               "question": item["question"], "decomposition": "rag",
               "reader_model": f"ollama {READER_OLLAMA_MODEL}",
               "retrieved_pages": [r["page"] for r in item["retrieved"]],
               "retrieval_scores": [r["score"] for r in item["retrieved"]],
               "context": ctx, "final_answer": ans}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"{item['id']} -> {ans[:160]}", flush=True)
print(f"traza -> {TRACE_OUT}")

## 3. Puntuar con el mismo scorer

Corre `rag_baseline/score.py` tal cual: misma exactitud numérica, misma definición de alucinación, misma regla Q3 con 1,596. Compara con el baseline en CPU (2/12: Q6 y Q11).

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "score.py", "--trace", TRACE_OUT,
                    "--retrieval", RETRIEVAL_FILE,
                    "--out", REPO + "/rag_baseline/results_colab.md"],
                   cwd=REPO + "/rag_baseline")
print("exit:", r.returncode)

## Alternativa documentada (Hugging Face, no por defecto en Colab)

Si quieres el lector HF en GPU en vez de Ollama, corre el script original con el modelo local (hay que bajar Qwen2.5-1.5B-Instruct a `memo/models/` o apuntar `--model` a `Qwen/Qwen2.5-1.5B-Instruct` en el Hub):

```bash
python rag_baseline/answer.py --retrieval rag_baseline/retrieval.json --model Qwen/Qwen2.5-1.5B-Instruct --out rag_baseline/run_trace_hf.jsonl
```

Ojo: `answer.py` carga el modelo sin `device_map`, así que en Colab correría en CPU salvo que se adapte. Por eso el default en Colab es Ollama.

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El notebook 08 (RAGAS) verifica que existan estas trazas antes de medir: sin ellas no hay métricas.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('rag_baseline/retrieval.json', 'Retrieval denso top-5 (12 preguntas)', True), ('rag_baseline/run_trace_colab.jsonl', 'Traza del lector (12 respuestas)', True), ('rag_baseline/results_colab.md', 'Resultados RAG Colab', True)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)